# 보고서용: S2 Parser 결과 통계

In [15]:
# scripts/report_s2_parser_00_stats.py (예시)
import json
from collections import Counter

In [16]:
path = "../runs/s2_parser.ir.jsonl"
n_proto = 0
step_counts = []
qcg_counts = []
data_counts = []
warn_counter = Counter()

In [17]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        n_proto += 1
        nodes = r["nodes"]
        step_counts.append(sum(1 for n in nodes if n["type"] == "Step"))
        qcg_counts.append(sum(1 for n in nodes if n["type"] == "QCGate"))
        data_counts.append(sum(1 for n in nodes if n["type"] == "DataAnalysis"))
        for w in r.get("warnings", []):
            warn_counter[w.split()[0]] += 1  # prefix 카운트

In [18]:
print("n_protocols:", n_proto)
print("avg_steps:", sum(step_counts) / len(step_counts))
print("avg_qcgates:", sum(qcg_counts) / len(qcg_counts))
print("avg_data_nodes:", sum(data_counts) / len(data_counts))
print("warnings:", warn_counter)

n_protocols: 46
avg_steps: 15.891304347826088
avg_qcgates: 1.1521739130434783
avg_data_nodes: 0.9565217391304348
warnings: Counter({'added_placeholder_step': 366, 'max_nodes': 7})


In [ ]:
# placeholder가 많으면 → Task Miner는 괜찮아도, Methods 내용 자체가 부족하거나, 프롬프트/MAX_NODES 설정이 빡센 것.

# 보고서용: S2B Evidence Only 결과 통계

In [12]:
# scripts/report_s2b_evidence_00_quality.py
import json
from collections import Counter

path = "../runs/s2b_grounded.evidence_only.ir.jsonl"

n_steps = 0
n_with_ev = 0
domain_counter = Counter()
domain_with_ev = Counter()

In [13]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        domain = r.get("domain") or "unknown"
        nodes = r["nodes"]
        for n in nodes:
            if n.get("type") != "Step":
                continue
            n_steps += 1
            domain_counter[domain] += 1
            if n.get("evidence"):
                n_with_ev += 1
                domain_with_ev[domain] += 1

In [14]:
print("총 Step 수:", n_steps)
print("evidence 있는 Step 수:", n_with_ev, "(비율:", n_with_ev / max(1, n_steps), ")")

print("\n도메인별 coverage:")
for d in sorted(domain_counter.keys()):
    tot = domain_counter[d]
    ok = domain_with_ev[d]
    print(f"  {d:40s}  {ok}/{tot}  ({ok / max(1, tot):.3f})")

총 Step 수: 731
evidence 있는 Step 수: 11 (비율: 0.015047879616963064 )

도메인별 coverage:
  Biochemical & Molecular Functional Analysis  1/64  (0.016)
  Bioimaging Technologies                   1/66  (0.015)
  Cell Biology & Culture                    0/75  (0.000)
  Genomics Technologies                     2/76  (0.026)
  Immunological Techniques                  3/80  (0.037)
  Microbiology & Virology                   3/68  (0.044)
  Model Organism-Specific Techniques        0/60  (0.000)
  Molecular Biology Techniques              0/80  (0.000)
  Neuroscience Methods                      1/82  (0.012)
  Plant Science & Technology                0/80  (0.000)


# 보고서용: IR Evidence Consistency 결과 통계

In [19]:
path = "../eval/ir_evidence_consistency.jsonl"
total = 0
supported = 0

In [20]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        params = r["params"]
        sup = r["check"]["supported_params"]
        uns = r["check"]["unsupported_params"]
        total += len(params)
        supported += len(sup)

In [21]:

print("총 파라미터 수:", total)
print("지원되는 파라미터 수:", supported)
print("지원 비율:", supported / max(1, total))

총 파라미터 수: 1162
지원되는 파라미터 수: 1138
지원 비율: 0.9793459552495697


In [30]:
path = "../eval/ir_evidence_consistency.no_ev.jsonl"
total = 0
supported = 0

In [31]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        params = r["params"]
        sup = r["check"]["supported_params"]
        uns = r["check"]["unsupported_params"]
        total += len(params)
        supported += len(sup)

In [32]:

print("총 파라미터 수:", total)
print("지원되는 파라미터 수:", supported)
print("지원 비율:", supported / max(1, total))

총 파라미터 수: 1162
지원되는 파라미터 수: 1141
지원 비율: 0.9819277108433735


# 보고서용: parameter 근거 품질 결과 통계

In [26]:
import json
import pandas as pd

rows = []
with open("../runs/verify_cov_params.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

# 전체 비율
print(df["verdict"].value_counts(normalize=True))

verdict
supported      0.783333
unsupported    0.128333
ambiguous      0.088333
Name: proportion, dtype: float64


In [28]:
# 프로토콜별 supported 비율
per_proto = df.groupby("protocol_id")["verdict"].value_counts(normalize=True).unstack().fillna(0)
print(per_proto)

verdict            ambiguous  supported  unsupported
protocol_id                                         
Bio-protocol-1010        0.0   1.000000     0.000000
Bio-protocol-1111        0.0   1.000000     0.000000
Bio-protocol-1174        0.0   1.000000     0.000000
Bio-protocol-1213        0.0   0.947368     0.052632
Bio-protocol-1250        0.0   0.894737     0.105263
Bio-protocol-1437        1.0   0.000000     0.000000
Bio-protocol-1537        1.0   0.000000     0.000000
Bio-protocol-1542        0.0   0.937500     0.062500
Bio-protocol-1611        0.0   0.948276     0.051724
Bio-protocol-1716        0.0   0.944444     0.055556
Bio-protocol-1784        0.0   1.000000     0.000000
Bio-protocol-1836        0.0   0.777778     0.222222
Bio-protocol-1861        0.0   0.884615     0.115385
Bio-protocol-2001        0.0   0.818182     0.181818
Bio-protocol-2096        0.0   0.881579     0.118421
Bio-protocol-2219        0.0   0.841270     0.158730
Bio-protocol-2302        0.0   0.809524     0.

In [38]:
df[df["protocol_id"] == "Bio-protocol-1437"]

,protocol_id,node_id,param_index,name,value,unit,verdict,evidence_span,llm_raw
293,Bio-protocol-1437,S2,0,embryonic day,NaN,None,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
294,Bio-protocol-1437,S3,0,antibody dilution,1.00,200,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
295,Bio-protocol-1437,S4,0,PFA fixation time 4C,2.00,h,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
296,Bio-protocol-1437,S4,1,PFA fixation time 37C,20.00,min,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
297,Bio-protocol-1437,S4,2,TCA fixation time,20.00,min,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
298,Bio-protocol-1437,S4,3,blocking time,3.00,h,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
299,Bio-protocol-1437,S4,4,primary antibody incubation time,NaN,None,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
300,Bio-protocol-1437,S4,5,secondary antibody incubation time,NaN,None,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
301,Bio-protocol-1437,S4,6,phalloidin concentration,0.30,μg/ml,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"
302,Bio-protocol-1437,S4,7,Hoechst concentration,5.00,μg/ml,ambiguous,,"{""verdict"": ""ambiguous"", ""evidence_span"": """"}"


In [35]:
# 예: unsupported 이면서 evidence_span 이 빈 문자열인 케이스만 모아보기
suspicious = df[(df["verdict"] == "unsupported") & (df["evidence_span"] == "")]
suspicious[suspicious["protocol_id"] == "Bio-protocol-1836"]

,protocol_id,node_id,param_index,name,value,unit,verdict,evidence_span,llm_raw
516,Bio-protocol-1836,S1,0,stage,NaN,None,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
520,Bio-protocol-1836,S6,1,homology_arm_length,1000.0,bp,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
